****Importing Kaggle Google API keys****

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GOOGLE_API_KEY")

In [2]:
import os
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Setup and authentication complete.


# ✈️ Travel Planner & Expense Assistant — Kaggle Capstone Project  
### 📚 *Project Track: Enterprise Agents*

---

## 🌟 Project Pitch

### **Problem Statement**  
Planning a trip often turns into a mess of tabs, notes, and spreadsheets:

- searching daily for local events, exhibitions, and activities  
- building a reasonable itinerary from scratch  
- budgeting the trip 
- digging through blogs and travel guides for trustworthy recommendations  
- creating summaries and files to share with the group  

Doing all of this manually is time-consuming, inconsistent, and prone to mistakes.

---

## 💡 Proposed Solution  
A **multi-agent system** that supports the traveler before and during the trip:

- 🔎 discovers relevant events, exhibitions, and things to do for selected dates  
- 🗺️ generates a structured daily itinerary automatically  
- 💰 finds accomodation based on budget
- 📝 analyzes blog posts and travel content to rank top recommendations  
- 📄 produces a polished final PDF report combining all trip details  

---

## 🎯 Value & Impact  
- **Instant organization:** itinerary and schedule generated in seconds  
- **Clear group finances:** every cost tracked and aggregated properly  
- **Reliable insights:** high-quality recommendations extracted and ranked  
- **Shareable outputs:** useful files for friends or trip documentation  
- **Fully integrated:** a single assistant coordinating multiple subtasks  

---

## 🧩 Agent Architecture

This system operates through a **4-phase multi-agent pipeline**, where each specialized sub-agent performs a distinct role and passes structured data to the next stage.

| **Phase** | **Agent** | **Purpose & Function** |
| :--: | :-- | :-- |
| **1️⃣** | ✈️ **Flight Price Comparison Agent** | Searches for the lowest round-trip flight prices from the origin city to each candidate destination using `google_search`. Outputs a clean JSON with `{city, estimated_flight_cost}` pairs. |
| **2️⃣** | 💰 **Travel & Budgeting Agent** | Orchestrates two internal agents:<br>• `BudgetEstimatorAgent`: estimates accommodation, activity, food, and transport costs per person.<br>• `CitySelectionAgent`: selects the cheapest destination within budget.<br>Outputs the final selected city JSON. |
| **3️⃣** | 🧳 **Content Insights Agent** | Analyzes travel blogs and local sources using `google_search`, generating **5 curated daily travel tips** for each day of the trip. Outputs structured JSON with `daily_tips`. |
| **4️⃣** | 🧾 **Travel Report Agent** | Combines itinerary, costs, and insights into a beautiful **Markdown-formatted travel report** — including summary, day-by-day plan, and travel tips. |

---

### ✅ Output Flow

```text
User Query
   ↓
Flight Price Comparison
   ↓
Travel & Budgeting (Estimation + City Selection)
   ↓
Content Insights
   ↓
Final Markdown Travel Report

---

In [3]:
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, ToolContext
from google.adk.code_executors import BuiltInCodeExecutor
from datetime import datetime, timedelta
from google import genai
from google.adk.runners import InMemoryRunner
from google.adk.agents import LlmAgent,SequentialAgent,Agent
from google.adk.tools import google_search
from google.adk.models.google_llm import Gemini
import json
import pandas as pd
from typing import Dict, Any
from google.adk.tools import agent_tool
import re
from google.adk.runners import Runner
from google.genai import types
import json
print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### Configuring retry option to avoid temporary service limits

In [4]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

### **✈️ Flight Price Comparison Agent**

| Field | Description |
| :--- | :--- |
| **Role** | Specialized researcher for flight costs. |
| **Workflow** | Analyzes candidates and dates; uses `Google Search` to find the estimated lowest round-trip flight price for *each* candidate city from the origin destination. |
| **Input** | User's full request (dates, budget, city candidates,origin city). |
| **Output** | Clean JSON array of `{city: cost}` pairs (e.g., `[{"city": "Lisbon", "estimated_flight_cost": "€180"}, ...]`). |

In [5]:
search_flights_agent = LlmAgent(
    name="FlightPriceComparisonAgent",
    model=Gemini(model="gemini-2.5-flash", retry_options=retry_config),
    instruction="""
    You are a Flight Price Comparison Specialist. Your sole task is to search for estimated round-trip flight prices 
    for the provided list of candidate destinations and return the data in a structured JSON array. 
    
    IMPORTANT: You must only provide the raw flight price data (City and Cost). Do NOT perform any budget calculations, 
    select a final city, or include any extra descriptive fields.

    Workflow:
    1. PARAMETER EXTRACTION: Analyze the user's request to identify 'Origin', 'Destination Candidates' and 'Travel Dates' (start_date, end_date).
    2. FLIGHT SEARCH & COMPARISON: For EACH candidate city, use the 'google_search' tool to find the **lowest estimated price** for a round-trip flight from the origin.
    3. FINAL OUTPUT: Strictly return the comparison data as a JSON array of objects.
       
    The Final Output JSON Format MUST adhere exactly to this structure (return only the array):
    [
        {"city": "Lisbon", "estimated_flight_cost": "€180"},
        {"city": "Rome", "estimated_flight_cost": "€50"},
        {"city": "Madrid", "estimated_flight_cost": "€250"}
    ]
    """,
    tools=[google_search],
    output_key="flight_comparison_data" # Key used by the next agent (Phase 2)
)
print("✅ FlightPriceComparisonAgent created (Phase 1).")

✅ FlightPriceComparisonAgent created (Phase 1).


### **💰 Budget Estimation Agent**

| Field | Description |
| :--- | :--- |
| **Role** | Specialized cost estimator for each candidate city. |
| **Workflow** | Calculates total estimated **trip cost per person** for each destination using average prices for accommodation, food, local transport, activities, and flight costs. Performs web lookups with `Google Search` to refine realistic cost data. |
| **Input** | Trip details including dates, origin city, number of travelers, per-person budget, and travel preferences (e.g., wine tasting, museums, sightseeing). |
| **Output** | Structured JSON **list** of cities with detailed per-person cost breakdowns.<br>Example:<br>`[{"city": "Lisbon", "total_trip_estimate": "€630.00", "flight_cost": "€150.00", ...}, ...]` |
| **Key Responsibilities** | - Estimate average accommodation cost per night and per person.<br>- Include realistic daily budgets for food, local transport, and activities.<br>- Return a complete itinerary with per-day cost breakdowns and event lists. |
| **Output Format** | JSON list — each object represents **one city**, with nested details for itinerary, activities, and total per-person costs. |


In [6]:
import json, sys
from typing import Any, Dict, List

def get_cheapest_city(cities_data_json: str, total_budget: float) -> Dict[str, Any]:
    """
    Selects the cheapest city JSON from a list of complete city data.
    Each city entry is a full JSON (with city, itinerary, total_trip_estimate, etc.).
    Adds `is_within_budget` flag based on whether the city fits the budget.
    """
    try:
        cities_data: List[Any] = json.loads(cities_data_json)
    except json.JSONDecodeError:
        return {"status": "error", "message": "Invalid JSON input for cities_data_json."}

    if cities_data and isinstance(cities_data[0], str):
        cities_data = [json.loads(c) for c in cities_data]

    def parse_euros(value: str) -> float:
        try:
            return float(value.replace("€", "").replace(",", "").strip())
        except Exception:
            return sys.float_info.max

    for c in cities_data:
        c["_numeric_estimate"] = parse_euros(c.get("total_trip_estimate", "inf"))


    cities_data.sort(key=lambda x: x["_numeric_estimate"])

    selected_city = None
    for c in cities_data:
        if c["_numeric_estimate"] <= total_budget:
            selected_city = c
            selected_city["is_within_budget"] = True
            break
    if not selected_city:
        selected_city = cities_data[0]
        selected_city["is_within_budget"] = False

    for c in cities_data:
        c.pop("_numeric_estimate", None)

    return selected_city


In [7]:
# Testing the function: expected output -> Lisbon
dummy_data = json.dumps([
    {
        "city": "Lisbon",
        "start_date": "2025-11-21",
        "end_date": "2025-11-23",
        "total_trip_estimate": "€630.00",
        "flight_cost": "€150.00",
        "accommodation_suggestion": {"cost_per_night": "€100.00", "total_accommodation_cost": "€300.00"},
        "itinerary_with_budget": [
            {
                "date": "2025-11-21",
                "daily_cost_estimate": "€210.00",
                "cost_breakdown": {"food": "€50.00", "local_transport": "€10.00", "activities_total": "€30.00"},
                "events": [
                    {"event": "Wine Tasting Tour", "time": "14:00", "estimated_cost": "€30.00"},
                    {"event": "Museum Visit", "time": "10:00", "estimated_cost": "€10.00"},
                    {"event": "Sightseeing", "time": "16:00", "estimated_cost": "€20.00"}
                ]
            },
            {
                "date": "2025-11-22",
                "daily_cost_estimate": "€210.00",
                "cost_breakdown": {"food": "€50.00", "local_transport": "€10.00", "activities_total": "€30.00"},
                "events": [
                    {"event": "Boat Tour on the Tagus", "time": "11:00", "estimated_cost": "€15.00"},
                    {"event": "Local Wine & Cheese Tasting", "time": "18:00", "estimated_cost": "€15.00"}
                ]
            },
            {
                "date": "2025-11-23",
                "daily_cost_estimate": "€210.00",
                "cost_breakdown": {"food": "€50.00", "local_transport": "€10.00", "activities_total": "€30.00"},
                "events": [
                    {"event": "Historic Tram Ride", "time": "10:30", "estimated_cost": "€10.00"},
                    {"event": "Sightseeing Alfama District", "time": "15:00", "estimated_cost": "€20.00"}
                ]
            }
        ]
    },
    {
        "city": "Rome",
        "start_date": "2025-11-21",
        "end_date": "2025-11-23",
        "total_trip_estimate": "€860.00",
        "flight_cost": "€180.00",
        "accommodation_suggestion": {"cost_per_night": "€120.00", "total_accommodation_cost": "€360.00"},
        "itinerary_with_budget": [
            {
                "date": "2025-11-21",
                "daily_cost_estimate": "€286.00",
                "cost_breakdown": {"food": "€60.00", "local_transport": "€10.00", "activities_total": "€80.00"},
                "events": [
                    {"event": "Colosseum Tour", "time": "10:00", "estimated_cost": "€25.00"},
                    {"event": "Wine Tasting", "time": "17:00", "estimated_cost": "€55.00"}
                ]
            },
            {
                "date": "2025-11-22",
                "daily_cost_estimate": "€287.00",
                "cost_breakdown": {"food": "€60.00", "local_transport": "€10.00", "activities_total": "€80.00"},
                "events": [
                    {"event": "Vatican Museums", "time": "09:00", "estimated_cost": "€25.00"},
                    {"event": "Sightseeing Trastevere", "time": "16:00", "estimated_cost": "€30.00"}
                ]
            },
            {
                "date": "2025-11-23",
                "daily_cost_estimate": "€287.00",
                "cost_breakdown": {"food": "€60.00", "local_transport": "€10.00", "activities_total": "€80.00"},
                "events": [
                    {"event": "Roman Forum Visit", "time": "10:00", "estimated_cost": "€20.00"},
                    {"event": "Dinner with Wine Pairing", "time": "19:00", "estimated_cost": "€60.00"}
                ]
            }
        ]
    },
    {
        "city": "Madrid",
        "start_date": "2025-11-21",
        "end_date": "2025-11-23",
        "total_trip_estimate": "€720.00",
        "flight_cost": "€160.00",
        "accommodation_suggestion": {"cost_per_night": "€110.00", "total_accommodation_cost": "€330.00"},
        "itinerary_with_budget": [
            {
                "date": "2025-11-21",
                "daily_cost_estimate": "€240.00",
                "cost_breakdown": {"food": "€55.00", "local_transport": "€10.00", "activities_total": "€35.00"},
                "events": [
                    {"event": "Museo del Prado", "time": "10:00", "estimated_cost": "€15.00"},
                    {"event": "Tapas & Wine Tour", "time": "19:00", "estimated_cost": "€20.00"}
                ]
            },
            {
                "date": "2025-11-22",
                "daily_cost_estimate": "€240.00",
                "cost_breakdown": {"food": "€55.00", "local_transport": "€10.00", "activities_total": "€35.00"},
                "events": [
                    {"event": "Royal Palace Visit", "time": "11:00", "estimated_cost": "€15.00"},
                    {"event": "Wine Tasting", "time": "16:00", "estimated_cost": "€20.00"}
                ]
            },
            {
                "date": "2025-11-23",
                "daily_cost_estimate": "€240.00",
                "cost_breakdown": {"food": "€55.00", "local_transport": "€10.00", "activities_total": "€35.00"},
                "events": [
                    {"event": "Retiro Park Sightseeing", "time": "10:00", "estimated_cost": "€10.00"},
                    {"event": "Flamenco Show", "time": "20:00", "estimated_cost": "€25.00"}
                ]
            }
        ]
    }
])

result = get_cheapest_city(dummy_data, total_budget=800)
print(json.dumps(result, indent=4, ensure_ascii=False))

{
    "city": "Lisbon",
    "start_date": "2025-11-21",
    "end_date": "2025-11-23",
    "total_trip_estimate": "€630.00",
    "flight_cost": "€150.00",
    "accommodation_suggestion": {
        "cost_per_night": "€100.00",
        "total_accommodation_cost": "€300.00"
    },
    "itinerary_with_budget": [
        {
            "date": "2025-11-21",
            "daily_cost_estimate": "€210.00",
            "cost_breakdown": {
                "food": "€50.00",
                "local_transport": "€10.00",
                "activities_total": "€30.00"
            },
            "events": [
                {
                    "event": "Wine Tasting Tour",
                    "time": "14:00",
                    "estimated_cost": "€30.00"
                },
                {
                    "event": "Museum Visit",
                    "time": "10:00",
                    "estimated_cost": "€10.00"
                },
                {
                    "event": "Sightseeing",
       

In [8]:
BudgetEstimatorAgent = Agent(
    name="BudgetEstimatorAgent",
    model="gemini-2.0-flash-lite",
    description="Estimates full trip costs for each city using Google Search and trip details.",
    instruction="""
You are the **Budget Estimation Agent**.
Your task is to estimate trip costs for each candidate city given:
- Travel dates (start_date, end_date)
- Origin (for example Luxembourg)
- Number of travelers (for example 3 people)
- Budget per person
- Preferences (wine tasting, museums, sightseeing)
- Flight costs per city (available in the state variable `flight_comparison_data`)

=== Steps ===
1️⃣ For **each city** (e.g. Lisbon, Rome, Madrid), estimate the following costs per person:
   - Accommodation per night using Google Search (e.g., “average hotel price per night in the city”).
   - Activity costs per person for the given preferences (wine tasting, museums, sightseeing).
   - Food (example €50 per person per day).
   - Local transport (example €10 per person per day).

2️⃣ Add the flight cost from `flight_comparison_data` and activity costs. Compute the "total_trip_estimate" per person as a sum of all the expenses: flight, accomodation, activities, food etc.


3️⃣ Return the output as a JSON **list**, where each entry corresponds to a single city with full trip details (each cost is per person ) , using this exact format:

[
  {
    "city": "Lisbon",
    "start_date": "2025-11-21",
    "end_date": "2025-11-23",
    "total_trip_estimate": "€630.00",
    "flight_cost": "€150.00",
    "accommodation_suggestion": {
        "cost_per_night": "€100.00",
        "total_accommodation_cost": "€300.00"
    },
    "Total_trip_costs": "€630.00",
    "itinerary_with_budget": [
        {
            "date": "2025-11-21",
            "daily_cost_estimate": "€210.00",
            "cost_breakdown": {
                "food": "€50.00",
                "local_transport": "€10.00",
                "activities_total": "€30.00"
            },
            "events": [
                {"event": "Wine Tasting Tour", "time": "14:00", "estimated_cost": "€30.00"}
            ]
        },
        {
            "date": "2025-11-22",
            "daily_cost_estimate": "€210.00",
            "cost_breakdown": {
                "food": "€50.00",
                "local_transport": "€10.00",
                "activities_total": "€30.00"
            },
            "events": [
                {"event": "Wine Tasting Tour", "time": "14:00", "estimated_cost": "€30.00"}
            ]
        },...
    ]
  }
]
    """,
    tools=[google_search],
)


CitySelectionAgent = Agent(
    name="CitySelectionAgent",
    model="gemini-2.0-flash-lite",
    description="Selects the cheapest city JSON based on total_trip_estimate and budget.",
    instruction="""
You are the **City Selection Agent**.

Input: A JSON array containing full trip data for multiple cities (see format from BudgetEstimatorAgent).

Goal:
- Select the city with the **lowest total_trip_estimate** that fits within the provided total budget (for example, €800 total per person).
- If all cities exceed the budget, select the cheapest one anyway.

You must use the Python function `get_cheapest_city(cities_data_json, total_budget)` to perform the filtering.

The function will return a **single JSON object** corresponding to the selected city.
It should retain the original structure and add one new field:
  `"is_within_budget": true` or `false`

=== Example Final Output ===
{
    "city": "Lisbon",
    "start_date": "2025-11-21",
    "end_date": "2025-11-23",
    "total_trip_estimate": "€630.00",
    "flight_cost": "€150.00",
    "accommodation_suggestion": {
        "cost_per_night": "€100.00",
        "total_accommodation_cost": "€300.00"
    },
    "Total_trip_costs": "€630.00",
    "itinerary_with_budget": [
        {
            "date": "2025-11-21",
            "daily_cost_estimate": "€210.00",
            "cost_breakdown": {
                "food": "€50.00",
                "local_transport": "€10.00",
                "activities_total": "€30.00"
            },
            "events": [
                {"event": "Wine Tasting Tour", "time": "14:00", "estimated_cost": "€30.00"}
            ]
        },
        {
            "date": "2025-11-22",
            "daily_cost_estimate": "XX.00",
            "cost_breakdown": {
                "food": "€XX.00",
                "local_transport": "€XX.00",
                "activities_total": "€XX.00"
            },
            "events": [
                {"event": "XX", "time": "XX:00", "estimated_cost": "€XX.00"}
            ]
        },..
    ],
    "is_within_budget": true
}
    """,
    tools=[get_cheapest_city],
)



TravelAndBudgetingAgent = Agent(
    name="TravelAndBudgetingAgent",
    model=Gemini(model="gemini-2.5-flash-lite"),
    instruction="""
You are the Final Orchestrator Agent.

Steps:
1️⃣ Call the `BudgetEstimatorAgent` to generate a JSON list of trip data for Lisbon, Rome, and Madrid.
2️⃣ Then call the `CitySelectionAgent` with that JSON list and the total budget (€800) to select the final city.
3️⃣ Return only the final city's full JSON (no explanation text).
    """,
    tools=[
        agent_tool.AgentTool(agent=BudgetEstimatorAgent, skip_summarization=True),
        agent_tool.AgentTool(agent=CitySelectionAgent, skip_summarization=True),
    ],
    output_key="final_budget_json" 
)

print("✅ TravelAndBudgetingAgent created (final orchestration setup). (Phase 2)")


✅ TravelAndBudgetingAgent created (final orchestration setup). (Phase 2)


### Testing the SubAgent with dummy flights data

In [9]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import json

APP_NAME = "TravelPlannerApp"
USER_ID = "user_default"
SESSION_ID = "travel-session"

session_service = InMemorySessionService()
runner = Runner(agent=TravelAndBudgetingAgent, app_name=APP_NAME, session_service=session_service)

print("✅ Runner ready for TravelAndBudgetingAgent!")


async def run_single_query(runner_instance, query, session_name):
    print(f"\n🚀 Starting session: {session_name}")
    print(f"🧑 User > {query}")

    try:
        await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_name
        )
    except Exception:
        await session_service.get_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_name
        )

    state_delta = {
        "flight_comparison_data": [
            {"city": "Lisbon", "flight_cost": 150},
            {"city": "Rome", "flight_cost": 180},
            {"city": "Madrid", "flight_cost": 160},
        ]
    }

    user_message = types.Content(role="user", parts=[types.Part(text=query)])

    full_output = ""

    async for event in runner_instance.run_async(
        user_id=USER_ID,
        session_id=session_name,
        new_message=user_message,
        state_delta=state_delta
    ):
        if event.content and getattr(event.content, "parts", None):
            for part in event.content.parts:
                if getattr(part, "function_response", None):
                    print("\n" + "=" * 30)
                    print("🏁 FINAL MODEL OUTPUT (formatted):")
        
                    try:

                        response_data = getattr(part.function_response, "response", {})
                        if isinstance(response_data, dict):
                            result_text = response_data.get("result", "")
                        else:
                            result_text = str(response_data)
        
                        clean_json = (
                            result_text.replace("```json", "")
                            .replace("```", "")
                            .strip()
                        )
        

                        json_start = clean_json.find("[")
                        json_end = clean_json.rfind("]") + 1
                        json_str = clean_json[json_start:json_end]
                        data = json.loads(json_str)
        

                        print("\n===== Options with prices =====")
                        for c in data:
                            print(f"- {c['city']}: {c['total_trip_estimate']}")
        
                        # ✅ Seleziona città più economica
                        def parse_euro(v):
                            return float(v.replace("€", "").replace(",", "").strip())
        
                        cheapest = min(data, key=lambda x: parse_euro(x["total_trip_estimate"]))
        
                        print("\n===== Selected city (cheapest) =====")
                        print(json.dumps(cheapest, indent=4, ensure_ascii=False))
        
                    except Exception as e:
                        print(f"⚠️ Error parsing JSON: {e}")
                        print("Raw output:")
                        print(part.function_response)
                    print("=" * 30)

await run_single_query(
    runner,
    "Plan a 3-day trip from 2025-11-21 to 2025-11-23 departing from Luxembourg. "
    "Candidate cities are Lisbon, Rome, and Madrid. Total budget is 800 euros for three people. "
    "Preferences: wine tasting, museums, sightseeing.",
    SESSION_ID
)


✅ Runner ready for TravelAndBudgetingAgent!

🚀 Starting session: travel-session
🧑 User > Plan a 3-day trip from 2025-11-21 to 2025-11-23 departing from Luxembourg. Candidate cities are Lisbon, Rome, and Madrid. Total budget is 800 euros for three people. Preferences: wine tasting, museums, sightseeing.



🏁 FINAL MODEL OUTPUT (formatted):

===== Options with prices =====
- Lisbon: €630.00
- Rome: €710.00
- Madrid: €690.00

===== Selected city (cheapest) =====
{
    "city": "Lisbon",
    "start_date": "2025-11-21",
    "end_date": "2025-11-23",
    "total_trip_estimate": "€630.00",
    "flight_cost": "€150.00",
    "accommodation_suggestion": {
        "cost_per_night": "€100.00",
        "total_accommodation_cost": "€300.00"
    },
    "Total_trip_costs": "€630.00",
    "itinerary_with_budget": [
        {
            "date": "2025-11-21",
            "daily_cost_estimate": "€210.00",
            "cost_breakdown": {
                "food": "€50.00",
                "local_transport": "€10.00",
                "activities_total": "€30.00"
            },
            "events": [
                {
                    "event": "Wine Tasting Tour",
                    "time": "14:00",
                    "estimated_cost": "€30.00"
                }
            ]
        },
        {
        

### **🧳 Content Insights Agent**

| Field | Description |
| :--- | :--- |
| **Role** | Travel insights specialist — creates contextual daily travel tips based on destination and preferences. |
| **Workflow** | <ol><li>Extract <code>city</code>, <code>start_date</code>, <code>end_date</code>, and <code>preferences</code>.</li><li>Use <code>google_search</code> to find travel blogs, local guides, or hidden gems for the destination.</li><li>Select the most relevant source.</li><li>Generate 5 curated daily travel tips per day.</li></ol> |
| **Input** | Final selected city JSON + user preferences. |
| **Output** | Structured JSON with daily tips:<br><br>
<code>{
&nbsp;&nbsp;"city": "Lisbon",<br>
&nbsp;&nbsp;"start_date": "2025-11-21",<br>
&nbsp;&nbsp;"end_date": "2025-11-23",<br>
&nbsp;&nbsp;"daily_tips": {<br>
&nbsp;&nbsp;&nbsp;&nbsp;"2025-11-21": ["Tip 1", "Tip 2", "Tip 3", "Tip 4", "Tip 5"],<br>
&nbsp;&nbsp;&nbsp;&nbsp;"2025-11-22": ["Tip 1", "Tip 2", "Tip 3", "Tip 4", "Tip 5"],<br>
&nbsp;&nbsp;&nbsp;&nbsp;"2025-11-23": ["Tip 1", "Tip 2", "Tip 3", "Tip 4", "Tip 5"]<br>
&nbsp;&nbsp;}<br>
}</code> |


In [10]:
content_insights_agent = LlmAgent(
    name="ContentInsightsAgent",
    model=Gemini(model="gemini-2.5-flash", retry_options=retry_config),
    instruction="""
    You are a travel insights specialist. Your task is to generate useful daily travel tips based on the destination, dates, and preferences provided in the input (which includes the final city selected by the previous agent).

    Workflow:
    1. INPUT EXTRACTION: The input contains the final city, dates, and user preferences. Extract the 'city', 'start_date', 'end_date', and 'preferences'.
    2. RESEARCH: Use 'google_search' with queries like: "<city> travel blog", "<city> hidden gems", etc.
    3. SCORING & SELECTION: Score collected sources and select the **top-ranked source**.
    4. TIPS GENERATION: From this top source, generate a curated set of **5 must-know tips per day of the trip**. The tips must be concrete, practical, and tailored to the destination.
    5. FINAL OUTPUT: Return the results strictly as a JSON object.
    
    Format:
    {
      "city": "<city>",
      "start_date": "<start_date>",
      "end_date": "<end_date>",
      "daily_tips": {
        "<date1>": ["Tip 1", "Tip 2", "Tip 3", "Tip 4", "Tip 5"],
        // ... for each day
      }
    }
    DO NOT PRINT THE JSON.
    """,
    tools=[google_search],
    output_key="daily_insights_json" # Key used by the final agent (Phase 4)
)
print("✅ ContentInsightsAgent created (Phase 3).")

✅ ContentInsightsAgent created (Phase 3).


### 🗺️ Travel Report Agent

| **Field** | **Description** |
| :--- | :--- |
| **Role** | Generates a final, beautifully formatted **Markdown travel report** combining all previous outputs (budget, itinerary, and tips). |
| **Workflow** | <ol><li>Merge `final_budget_json` and `daily_insights_json`.</li><li>Extract key trip details (city, dates, total cost, activities, and tips).</li><li>Format the result into a professional **travel report** with emojis, headings, and clear structure.</li></ol> |
| **Input** | - Trip & budget data from Phase 2 (`final_budget_json`)<br>- Daily tips & insights from Phase 3 (`daily_insights_json`) |
| **Output** | A fully formatted **Markdown travel report — not JSON —** containing:<br><br>```markdown<br>✈️ 3-Day Trip to Lisbon (Nov 21–23, 2025)<br><br>## 🏖️ Trip Summary & Budget Overview<br>- **Destination:** Lisbon 🇵🇹<br>- **Total Trip Estimate:** €630.00<br>- **Flight Cost:** €150.00<br>- **Accommodation:** €100/night (€300 total)<br>- **Budget Status:** ✅ Within Budget<br><br>## 📅 Daily Itinerary & Cost Breakdown<br>- **Day 1:** Wine Tasting (14:00, €30)<br>- **Day 2:** Museum Visit (10:00, €15)<br><br>## 💡 Must-Know Daily Tips<br>- **Nov 21:** “Book your wine tasting early.”<br>- **Nov 22:** “Try local pastel de nata.”<br>``` |


In [11]:
travel_report_agent = LlmAgent(
    name="TravelReportAgent",
    model=Gemini(model="gemini-2.5-flash", retry_options=retry_config),
    instruction="""
    You are the **Final Report Generator**.

    Your job is to receive all structured data from the previous pipeline steps
    — the finalized itinerary & budget (from `{final_budget_json}`) and the daily insights (from `{daily_insights_json}`) —
    and create a **beautiful, well-structured Markdown travel report**.

    === Workflow ===
    1️⃣ DATA FUSION:
       - Parse both JSONs (`final_budget_json` and `daily_insights_json`).
       - Extract key fields: city, dates, total costs, itinerary details, daily tips.

    2️⃣ REPORT GENERATION:
       Write a clean Markdown report using clear formatting and emojis where appropriate.
       Include these sections:
       - **🏖️ Trip Summary & Budget Overview:** destination, travel dates, total cost, flights, accommodation, budget status (within/over).
       - **📅 Daily Itinerary & Activities:** per day list of planned events, times, and estimated costs (formatted as lists or tables).
       - **💡 Must-Know Tips:** 5 curated tips per day from `{daily_insights_json}`.

    3️⃣ OUTPUT RULES:
       - Use only Markdown formatting (no JSON).
       - Make it engaging but concise, like a professional travel guide.
       - Return ONLY the final Markdown text.

    Example structure:

    # ✈️ 3-Day Trip to Lisbon (Nov 21–23, 2025)

    ## 🏖️ Trip Summary & Budget Overview
    - **Destination:** Lisbon 🇵🇹
    - **Total Trip Estimate:** €630.00
    - **Flight Cost:** €150.00
    - **Accommodation:** €100/night (€300 total)
    - **Budget Status:** ✅ Within Budget

    ## 📅 Daily Itinerary & Cost Breakdown
    **Day 1 (Nov 21):**
    - Wine Tasting Tour — 14:00 (€30)
    - Dinner & Sightseeing (€50)
    - Total Daily Estimate: €210

    **Day 2 (Nov 22):**
    - Museum Visit — 10:00 (€15)
    - Local Market Tour — 15:00 (€20)
    - Total Daily Estimate: €210

    ## 💡 Must-Know Daily Tips
    **Nov 21:**
    - Tip 1: Book your wine tasting early to avoid queues.
    - Tip 2: Try local pastel de nata from Manteigaria.
    ...
    """,
    output_key="final_markdown_report"
)

print("✅ TravelReportAgent created (Phase 4).")


✅ TravelReportAgent created (Phase 4).


In [12]:
root_agent = SequentialAgent(
    name="FullTripPlanningPipeline",
    sub_agents=[
        search_flights_agent,        # 1. Get Flight Prices
        TravelAndBudgetingAgent,  # 2. Select City, Calculate Budget & Itinerary
        content_insights_agent,      # 3. Get Daily Travel Tips
        travel_report_agent,         # 4. Compile Final Markdown Report
    ],
)
print("✅ Sequential Agent (4-Phase Pipeline) created.")

✅ Sequential Agent (4-Phase Pipeline) created.


## ▶️ Run the pipeline - putting everything together

In [13]:
# ============================================
# 🚀 RUNNER — Full Trip Planning Pipeline (Final Clean Version)
# ============================================

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "TravelPlannerApp"
USER_ID = "user_default"
SESSION_ID = "full-trip-session"

# ---- Setup runner for the full pipeline ----
session_service = InMemorySessionService()
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

print("✅ Runner ready for FullTripPlanningPipeline!")


async def run_full_pipeline(query: str):
    print(f"\n🧑 User > {query}")


    try:
        await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
        )
    except Exception:
        await session_service.get_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
        )

    user_message = types.Content(role="user", parts=[types.Part(text=query)])
    final_output = ""

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=user_message
    ):
        if event.content and getattr(event.content, "parts", None):
            for part in event.content.parts:
                # ✅ Filtra e mostra solo l'output del TravelReportAgent (il Markdown finale)
                if getattr(part, "text", None) and part.text.strip().startswith("# ✈️"):
                    final_output += part.text.strip() + "\n"

    print("\n==============================")
    print("🏁 FINAL TRAVEL REPORT:")
    print(final_output.strip())
    print("==============================")



await run_full_pipeline(
    "Plan a 3-day trip from 2025-11-21 to 2025-11-23 departing from Luxembourg. "
    "Candidate cities are Lisbon, Rome, and Madrid. Total budget is 800 euros per person. "
    "Preferences: wine tasting, museums, sightseeing."
)


✅ Runner ready for FullTripPlanningPipeline!

🧑 User > Plan a 3-day trip from 2025-11-21 to 2025-11-23 departing from Luxembourg. Candidate cities are Lisbon, Rome, and Madrid. Total budget is 800 euros per person. Preferences: wine tasting, museums, sightseeing.



🏁 FINAL TRAVEL REPORT:
# ✈️ 3-Day Trip to Lisbon (Nov 21–23, 2025)

## 🏖️ Trip Summary & Budget Overview
- **Destination:** Lisbon 🇵🇹
- **Travel Dates:** November 21–23, 2025
- **Total Trip Estimate:** €630.00
- **Flight Cost:** €42.00
- **Accommodation:** €100.00/night (€300.00 total for 3 nights)
- **Budget Status:** ✅ Within Budget (Total budget €800.00 per person)

## 📅 Daily Itinerary & Cost Breakdown

**Day 1 (Nov 21): Exploring Historic Lisbon**
- **Morning/Afternoon:** Focus on sightseeing and museums in the historic Alfama and Baixa districts. Suggested activities include exploring Alfama, visiting Castelo de São Jorge, riding Tram 28 (or 12/24), strolling Rua Augusta and Praça do Comércio, and visiting the National Museum of Ancient Art.
- **Evening:** Engage in a Wine Tasting Tour or explore local dining options.
- **Total Daily Estimate:** €210.00
  - Food: €50.00
  - Local Transport: €10.00
  - Activities (Wine Tasting Tour, Sightseeing & Museums): €100.00

**Day 2 (Nov 2